In [ ]:
import pandas as pd
import pickle
import os
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.feature_selection import SelectFromModel
from sklearn import metrics
from sklearn import preprocessing
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from inspect import signature
from matplotlib.ticker import FormatStrFormatter
from sklearn.model_selection import train_test_split
import matplotlib.lines as mlines
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn import metrics
import seaborn as sns
from sklearn.metrics import average_precision_score
from matplotlib.lines import Line2D
import scipy.io as sio
from sklearn.impute import KNNImputer
from scipy import stats
from sklearn.utils import resample
import umap
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from BorutaShap import BorutaShap, load_data
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import plot_confusion_matrix
from sklearn.multiclass import OneVsOneClassifier
from sklearn.feature_selection import SelectKBest, f_classif
import pickle
from sklearn.linear_model import SGDClassifier
import itertools
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

In [ ]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
    plt.figure(figsize=(8, 6),dpi=300)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names)
        plt.yticks(tick_marks, target_names)


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

# Load Data

In [ ]:
study_criteria_table = pd.read_excel(r"../medical_data/study_criteria_table_label_V6.xlsx")


In [ ]:
criteria_df = study_criteria_table 

In [ ]:
study_features_table = pd.read_csv(r"../eeg_data/study_features_table_v4.csv")

# Preprocess Data

In [ ]:
data_df = study_criteria_table[['FolderName','Predicted_Stage']].merge(study_features_table,on=['FolderName'])

In [ ]:
data_df = data_df.drop_duplicates(subset=['FolderName'])

In [ ]:
data_df

In [ ]:
dementia_df = pd.DataFrame(data=criteria_df[(criteria_df['Predicted_Stage']== 'Dementia' ) ]['FolderName'],columns=['FolderName'])
dementia_df['class'] = 1

MCI_df = pd.DataFrame(criteria_df[(criteria_df['Predicted_Stage']== 'MCI' )]['FolderName'],columns=['FolderName'])
MCI_df['class'] =2

nondementia_df = pd.DataFrame(criteria_df[(criteria_df['Predicted_Stage']== 'No Dementia' )]['FolderName'],columns=['FolderName'])
nondementia_df['class'] = 3

In [ ]:
print('Dementia:',len(dementia_df))
print('MCI:',len(MCI_df))
print('No Dementia:',len(nondementia_df))

In [ ]:
y = pd.concat([nondementia_df,MCI_df,dementia_df])
y = y.sample(frac=1)
y=y.reset_index(drop=True)
X = y.merge(data_df,on=['FolderName'],how='left')
X = X[X.columns[3:]]
y = y[y.columns[1]]

In [ ]:
X.to_csv('X_All.csv',index=False)
y.to_csv('y_All.csv',index=False)
#X = pd.read_csv('X_All.csv')
#y = pd.read_csv('y_All.csv')

# Logistic Regression

In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0,0],[0,0,0],[0,0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.zeros([0,3])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):
    f+=1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #imputer after scaler
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
    
    #scaler = preprocessing.StandardScaler().fit(X_test)
    #imputer = KNNImputer(n_neighbors=10).fit(X_test)
    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the model
    skb = SelectKBest(f_classif,k=350)
    fs_model = RandomForestClassifier(random_state=7,class_weight='balanced')
    model = LogisticRegression(random_state=7,penalty='elasticnet',solver='saga',class_weight='balanced',multi_class='multinomial' )
     
    pipeline = Pipeline(
    [ ("filter",skb),
        ("feature_selection", SelectFromModel(fs_model)),
        ("classification",model)])
    
    params = { 'feature_selection__threshold':[0.00005,0.0001,0.0005,0.001,0.002,0.003,0.004,0.005,0.007,0.01],
        'classification__C': [10**x for x in range(-3,5)],
        'classification__l1_ratio' : [0.5,0.6,0.7,0.8,0.9]}
    
    
    gd_search = GridSearchCV(pipeline, params, scoring='f1_macro', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_

    y_prob = best_model.predict_proba(X_test)
    y_pred =  best_model.predict(X_test)
    y_test_binarized =  label_binarize(y_test, classes=[1, 2, 3])
    
    auc = metrics.roc_auc_score(y_test_binarized, y_prob,multi_class = 'ovr')
    f1 = metrics.f1_score(y_test, y_pred,average='macro')
    
    # store the result
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred,average='macro')))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred,average='macro')))
    print('F1 Score : ' + str(f1))
    kappa = metrics.cohen_kappa_score(y_test,y_pred)
    print('Kappa : ' + str(kappa))
    kappa_all.append(kappa)
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[1, 2,3])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model
print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
#with open('DM_v_MCI_v_CN_LG_scores.pickle', 'wb') as f:
#    pickle.dump([y_test_all,y_prob_all,y_pred_all,cm_total], f)
with open('DM_v_MCI_v_CN_LG_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)

In [ ]:
#https://machinelearningmastery.com/precision-recall-and-f-measure-for-imbalanced-classification/
specificity = cm_total[2,2]/(cm_total[2,0]+cm_total[2,1] + cm_total[2,2])
sensitivity = (cm_total[0,0] + cm_total[1,1] )/((cm_total[0,0]+cm_total[0,1]+cm_total[0,2])+(cm_total[1,0]+cm_total[1,1]+cm_total[1,2]))
print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)

precision = metrics.precision_score(y_test_all, y_pred_all, labels=[1,2], average='micro')
print('Precision: %f' % precision)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)
f1_micro = metrics.f1_score(y_test_all,y_pred_all,average='micro')
print("F1 micro:",f1_micro)
f1_macro = metrics.f1_score(y_test_all,y_pred_all,average='macro')
print("F1 macro:",f1_macro)

plot_confusion_matrix(cm=cm_total,
                          target_names=['DM','MCI','CN'],
                          title='Confusion matrix',
                          normalize=True)


report= metrics.classification_report(y_test_all,y_pred_all)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='micro')
print("F1 micro:",f1_weighted)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='macro')
print("F1 macro:",f1_weighted)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
auc = metrics.roc_auc_score(y_test_all_binarized, y_prob_all,multi_class = 'ovr')
print("AUC:",auc)
print(report)
plt.savefig('../figures/Fig6_A.svg',format='svg')
plt.show()

In [ ]:
plt.figure(dpi=300)
labels = ['Dementia','MCI','Cognitively Normal']
colors = ['tab:Red','orange','tab:blue']
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
for i in range(3):
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.plot(fpr, tpr, label= 'Class ' + labels[i] + ' AUROC=%0.2f' % roc_auc,color=colors[i])

plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right",frameon=False)
plt.title('ROC Curve')
plt.savefig('../figures/Fig6_A.svg',format='svg')
plt.show()

In [ ]:
plt.figure(dpi=300)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
labels = ['Dementia','MCI','Cognitively Normal']
for i in range(3):
    precision, recall, thresholds = precision_recall_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    average_precision = average_precision_score(y_test_all_binarized[:,i],y_prob_all[:,i])
    plt.plot(recall, precision, label=labels[i] + ' Class '+'AUPRC = %0.2f' % average_precision,color=colors[i])
    if i == 2:
        break

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve'.format(
          average_precision))
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.legend(loc="upper right",frameon=False)
plt.savefig('../figures/Fig6_B.svg',format='svg')
plt.show()

In [ ]:
#Final Model
scaler = preprocessing.StandardScaler().fit(X)
X_processed = pd.DataFrame(data=scaler.transform(X),columns=X.columns)
imputer = KNNImputer(n_neighbors=10).fit(X_processed)
X_processed = pd.DataFrame(data=imputer.transform(X_processed),columns=X.columns)
    
final_model = LogisticRegression(random_state=7,C = 0.1 ,penalty='elasticnet',solver='saga',l1_ratio=0.8,class_weight = 'balanced')
final_model.fit(X_processed,y)


In [ ]:
target_names=['Dementia','MCI','Nondementia']
feat_with_weights = sorted(zip(final_model.coef_[0], list(X.columns)))
plt.figure(figsize=(20,10)) 
x_values = [r[1]for r in feat_with_weights[-20:]]
y_values = [r[0]for r in feat_with_weights[-20:]]
plt.barh(x_values,y_values)
plt.xticks(rotation='80')
plt.xlabel('Feature Weight')
plt.title('Important Features for ' + target_names[0])
plt.show()

In [ ]:
feat_with_weights = sorted(zip(final_model.coef_[1], list(X.columns)))
plt.figure(figsize=(20,10)) 
x_values = [r[1]for r in feat_with_weights[-20:]]
y_values = [r[0]for r in feat_with_weights[-20:]]
plt.barh(x_values,y_values)
plt.xticks(rotation='80')
plt.xlabel('Feature Weight')
plt.title('Important Features for ' + target_names[1])
plt.show()

In [ ]:
feat_with_weights = sorted(zip(final_model.coef_[2], list(X.columns)))
plt.figure(figsize=(20,10)) 
x_values = [r[1]for r in feat_with_weights[-20:]]
y_values = [r[0]for r in feat_with_weights[-20:]]
plt.barh(x_values,y_values)
plt.xticks(rotation='80')
plt.xlabel('Feature Weight')
plt.title('Important Features for ' + target_names[2])
plt.show()

# SVM

In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0,0],[0,0,0],[0,0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.zeros([0,3])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):
    f+=1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #imputer after scaler
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)

    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the model
    skb = SelectKBest(f_classif,k=350)
    fs_model = RandomForestClassifier(random_state=7,class_weight='balanced')   
    model = SGDClassifier(loss='hinge',random_state=7,class_weight='balanced')
    pipeline = Pipeline(
    [ ("filter",skb),
        ("feature_selection", SelectFromModel(fs_model)),
        ("classification",model)])
    params = {  'feature_selection__threshold':[0.00005,0.0001,0.0005,0.001,0.002,0.003,0.004,0.005,0.007,0.01],'classification__alpha': [10**x for x in range(-5,5)]}

     
    gd_search = GridSearchCV(pipeline, params, scoring='f1_macro', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_

    y_prob = best_model.decision_function(X_test)
    y_pred =  best_model.predict(X_test)

    y_test_binarized =  label_binarize(y_test, classes=[1, 2, 3])
    
    auc = metrics.roc_auc_score(y_test_binarized, y_prob,multi_class = 'ovr')
    f1 = metrics.f1_score(y_test, y_pred,average='macro')
    
    # store the result
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred,average='macro')))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred,average='macro')))
    print('F1 Score : ' + str(f1))
    kappa = metrics.cohen_kappa_score(y_test,y_pred)
    print('Kappa : ' + str(kappa))
    kappa_all.append(kappa)
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[1, 2,3])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model
print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
#with open('DM_v_MCI_v_CN_SVM_scores.pickle', 'wb') as f:
#    pickle.dump([y_test_all,y_prob_all,y_pred_all,cm_total], f)
with open('DM_v_MCI_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)

In [ ]:
#https://machinelearningmastery.com/precision-recall-and-f-measure-for-imbalanced-classification/
specificity = cm_total[2,2]/(cm_total[2,0]+cm_total[2,1] + cm_total[2,2])
sensitivity = (cm_total[0,0] + cm_total[1,1] )/((cm_total[0,0]+cm_total[0,1]+cm_total[0,2])+(cm_total[1,0]+cm_total[1,1]+cm_total[1,2]))
print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)

precision = metrics.precision_score(y_test_all, y_pred_all, labels=[1,2], average='micro')
print('Precision: %f' % precision)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)
f1_micro = metrics.f1_score(y_test_all,y_pred_all,average='micro')
print("F1 micro:",f1_micro)
f1_macro = metrics.f1_score(y_test_all,y_pred_all,average='macro')
print("F1 macro:",f1_macro)

plot_confusion_matrix(cm=cm_total,
                          target_names=['DM','MCI','CN'],
                          title='Confusion matrix',
                          normalize=True)
report= metrics.classification_report(y_test_all,y_pred_all)

f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='micro')
print("F1 micro:",f1_weighted)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='macro')
print("F1 macro:",f1_weighted)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)
#auc = metrics.roc_auc_score(y_test_binarized, y_prob,multi_class = 'ovr')
#print("AUC:",auc)
print(report)
plt.savefig('../figures/Fig6_B.svg',format='svg')
plt.show()

In [ ]:
plt.figure(dpi=300)
labels = ['Dementia','MCI','Cognitively Normal']
colors = ['tab:Red','orange','tab:blue']
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
for i in range(3):
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.plot(fpr, tpr, label= 'Class ' + labels[i] + ' AUROC=%0.2f' % roc_auc,color=colors[i])

plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right",frameon=False)
plt.title('ROC Curve')
plt.savefig('../figures/Fig6_A.svg',format='svg')
plt.show()

In [ ]:
plt.figure(dpi=300)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
labels = ['Dementia','MCI','Cognitively Normal']
for i in range(3):
    precision, recall, thresholds = precision_recall_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    average_precision = average_precision_score(y_test_all_binarized[:,i],y_prob_all[:,i])
    plt.plot(recall, precision, label=labels[i] + ' Class '+'AUPRC = %0.2f' % average_precision,color=colors[i])
    if i == 2:
        break

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve'.format(
          average_precision))
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.legend(loc="upper right",frameon=False)
plt.savefig('../figures/Fig6_B.svg',format='svg')
plt.show()

In [ ]:
#Final Model
scaler = preprocessing.StandardScaler().fit(X)
X_processed = pd.DataFrame(data=scaler.transform(X),columns=X.columns)
imputer = KNNImputer(n_neighbors=10).fit(X_processed)
X_processed = pd.DataFrame(data=imputer.transform(X_processed),columns=X.columns)

final_model = SVC(C = 0.1 ,kernel='linear',probability=True)
final_model = SelectFromModel(final_model,threshold=0.0001)
final_model.fit(X_processed,y)

# Random Forest 

In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0,0],[0,0,0],[0,0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.zeros([0,3])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):
    f+=1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #imputer after scaler
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
    
    #scaler = preprocessing.StandardScaler().fit(X_test)
    #imputer = KNNImputer(n_neighbors=10).fit(X_test)
    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the model
    model = RandomForestClassifier(random_state=7,class_weight='balanced')
    pipeline = Pipeline(
    [ ("feature_selection", SelectFromModel(model)),
        ("classification",model)])
    
    params = { 'feature_selection__threshold':[0.00005,0.0001,0.0005,0.001,0.005,0.01],
        'classification__max_depth': [3, 5, 7, 10,15,20,25,30,40],
        'classification__n_estimators' : [50, 100, 200],
        'classification__ccp_alpha': [0.01, 0.2,0.4,0.5, 0.1]}
    gd_search = GridSearchCV(pipeline, params, scoring='f1_macro', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_
    

    #classifier = OneVsRestClassifier(best_model)
    #y_prob = best_model.decision_function(X_test)
    y_prob = best_model.predict_proba(X_test)
    y_pred =  best_model.predict(X_test)
    y_test_binarized =  label_binarize(y_test, classes=[1, 2, 3])
    
    auc = metrics.roc_auc_score(y_test_binarized, y_prob,multi_class = 'ovr')
    f1 = metrics.f1_score(y_test, y_pred,average='macro')
    
    # store the result
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred,average='macro')))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred,average='macro')))
    print('F1 Score : ' + str(f1))
    kappa = metrics.cohen_kappa_score(y_test,y_pred)
    print('Kappa : ' + str(kappa))
    kappa_all.append(kappa)
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[1, 2,3])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model
print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
#with open('DM_v_MCI_v_CN_RF_scores.pickle', 'wb') as f:
#    pickle.dump([y_test_all,y_prob_all,y_pred_all,cm_total], f)
with open('DM_v_MCI_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)

In [ ]:
cm_total

In [ ]:
#https://machinelearningmastery.com/precision-recall-and-f-measure-for-imbalanced-classification/
specificity = cm_total[2,2]/(cm_total[2,0]+cm_total[2,1] + cm_total[2,2])
sensitivity = (cm_total[0,0] + cm_total[1,1] )/((cm_total[0,0]+cm_total[0,1]+cm_total[0,2])+(cm_total[1,0]+cm_total[1,1]+cm_total[1,2]))
print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)

precision = metrics.precision_score(y_test_all, y_pred_all, labels=[1,2], average='micro')
print('Precision: %f' % precision)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)
f1_micro = metrics.f1_score(y_test_all,y_pred_all,average='micro')
print("F1 micro:",f1_micro)
f1_macro = metrics.f1_score(y_test_all,y_pred_all,average='macro')
print("F1 macro:",f1_macro)
plot_confusion_matrix(cm=cm_total,
                          target_names=['DM','MCI','CN'],
                          title='Confusion matrix',
                          normalize=True)
report= metrics.classification_report(y_test_all,y_pred_all)


print(report)
plt.savefig('../figures/Fig6_C.svg',format='svg')
plt.show()

In [ ]:
plt.figure(dpi=300)
labels = ['Dementia','MCI','Cognitively Normal']
colors = ['tab:Red','orange','tab:blue']
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
for i in range(3):
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.plot(fpr, tpr, label= 'Class ' + labels[i] + ' AUROC=%0.2f' % roc_auc,color=colors[i])

plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right",frameon=False)
plt.title('ROC Curve')
plt.savefig('../figures/Fig6_A.svg',format='svg')
plt.show()

In [ ]:
plt.figure(dpi=300)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
labels = ['Dementia','MCI','Cognitively Normal']
for i in range(3):
    precision, recall, thresholds = precision_recall_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    average_precision = average_precision_score(y_test_all_binarized[:,i],y_prob_all[:,i])
    plt.plot(recall, precision, label=labels[i] + ' Class '+'AUPRC = %0.2f' % average_precision,color=colors[i])
    if i == 2:
        break

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve'.format(
          average_precision))
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.legend(loc="upper right",frameon=False)
plt.savefig('../figures/Fig6_B.svg',format='svg')
plt.show()

In [ ]:
#Final Model

scaler = preprocessing.StandardScaler().fit(X)
X_processed = pd.DataFrame(data=imputer.transform(X),columns=X.columns)
imputer = KNNImputer(n_neighbors=10).fit(X_processed)
X_processed = pd.DataFrame(data=scaler.transform(X_processed),columns=X.columns)

final_model = RandomForestClassifier(random_state=7,ccp_alpha = 0.01,max_depth= 7, n_estimators=200)
final_model = SelectFromModel(final_model,threshold=0.0001)
final_model.fit(X_processed,y)

In [ ]:
feature_importance_df = pd.DataFrame(columns=['feature','importance'])
feature_importance_df['feature'] = X.columns
feature_importance_df['importance'] = final_model.estimator_.feature_importances_
feature_importance_df
feature_importance_df.sort_values(by=['importance'],ascending=False)[:50]

# Figure

In [ ]:
with open('DM_v_MCI_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
fpr_micro, tpr_micro, thresholds = metrics.roc_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
roc_auc_micro = metrics.auc(fpr_micro, tpr_micro)
plt.plot(fpr_micro, tpr_micro, color='tab:red', label='SVM AUROC=%0.2f' % roc_auc_micro)

with open('DM_v_MCI_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
fpr_micro, tpr_micro, thresholds = metrics.roc_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
roc_auc_micro = metrics.auc(fpr_micro, tpr_micro)
plt.plot(fpr_micro, tpr_micro, color='green', label='Random Forest AUROC=%0.2f' % roc_auc_micro)

In [ ]:
#micro-average
plt.figure(dpi=300)
with open('DM_v_MCI_v_CN_LG_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
fpr_micro, tpr_micro, thresholds = metrics.roc_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
roc_auc_micro = metrics.auc(fpr_micro, tpr_micro)
plt.plot(fpr_micro, tpr_micro, color='tab:blue', label='Logistic Regression AUROC=%0.2f' % roc_auc_micro)

with open('DM_v_MCI_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
fpr_micro, tpr_micro, thresholds = metrics.roc_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
roc_auc_micro = metrics.auc(fpr_micro, tpr_micro)
plt.plot(fpr_micro, tpr_micro, color='tab:red', label='SVM AUROC=%0.2f' % roc_auc_micro)

with open('DM_v_MCI_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
fpr_micro, tpr_micro, thresholds = metrics.roc_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
roc_auc_micro = metrics.auc(fpr_micro, tpr_micro)
plt.plot(fpr_micro, tpr_micro, color='green', label='Random Forest AUROC=%0.2f' % roc_auc_micro)

plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right",frameon=False)
plt.title('ROC Curve')
plt.savefig('../figures/Fig6_A.svg',format='svg')
plt.show()


In [ ]:
#micro-average
plt.figure(dpi=300)
with open('DM_v_MCI_v_CN_LG_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
average_precision_micro = average_precision_score(y_test_all_binarized.ravel(), y_prob_all.ravel())
precision_micro, recall_micro, thresholds = precision_recall_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
plt.plot( recall_micro, precision_micro,color='tab:blue', label='Logistic Regression AUPRC=%0.2f' % average_precision)

with open('DM_v_MCI_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
average_precision_micro = average_precision_score(y_test_all_binarized.ravel(), y_prob_all.ravel())
precision_micro, recall_micro, thresholds = precision_recall_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
plt.plot( recall_micro, precision_micro,color='tab:red', label='SVM AUPRC=%0.2f' % average_precision_micro)

with open('DM_v_MCI_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
y_test_all_binarized = label_binarize(y_test_all, classes=[1, 2, 3])
average_precision_micro = average_precision_score(y_test_all_binarized.ravel(), y_prob_all.ravel())
precision_micro, recall_micro, thresholds = precision_recall_curve(y_test_all_binarized.ravel(), y_prob_all.ravel())
plt.plot( recall_micro, precision_micro,color='green', label='Random Forest AUPRC=%0.2f' % average_precision_micro)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.legend(loc="lower right",frameon=False)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.0])
plt.xlim([0.0, 1.0])
plt.title('Precision-Recall curve'.format(
          average_precision))
plt.savefig('../figures/Fig6_B.svg',format='svg')
plt.show()

In [ ]:
#with open('DM_v_MCI_v_CN_LG_scores.pickle', 'wb') as f:
#    pickle.dump([y_test_all,y_prob_all,y_pred_all,cm_total], f)


In [ ]:
#https://machinelearningmastery.com/precision-recall-and-f-measure-for-imbalanced-classification/

with open('DM_v_MCI_v_CN_LG_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
    
cm=cm_total
target_names=['DEM','MCI','CN']
title='Confusion matrix'
cmap=None
normalize=True


if cmap is None:
    cmap = plt.get_cmap('Blues')
    newcolors = np.vstack((cmap(np.linspace(0, 0.01, 60)),cmap(np.linspace(0, 1, 128)),
    cmap(np.linspace(0.99, 1, 70))))
    cmap = ListedColormap(newcolors)

if normalize:
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6, 5),dpi=300)
plt.imshow(cm, interpolation='nearest',vmin=0.0,vmax=1, cmap=cmap)
#plt.title(title)
plt.colorbar()

if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=0)
    plt.yticks(tick_marks, target_names)


thresh = cm.max() / 1.5 if normalize else cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    if normalize:
        plt.text(j, i, "{:0.3f}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    else:
        plt.text(j, i, "{:,}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')

plt.savefig('../figures/Fig6_A.svg',format='svg')
plt.show()

In [ ]:
#https://machinelearningmastery.com/precision-recall-and-f-measure-for-imbalanced-classification/

with open('DM_v_MCI_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
    
cm=cm_total
target_names=['DEM','MCI','CN']
title='Confusion matrix'
cmap=None
normalize=True


if cmap is None:
    cmap = plt.get_cmap('Blues')
    newcolors = np.vstack((cmap(np.linspace(0, 0.01, 60)),cmap(np.linspace(0, 1, 128)),
    cmap(np.linspace(0.99, 1, 70))))
    cmap = ListedColormap(newcolors)

if normalize:
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6, 5),dpi=300)
plt.imshow(cm, interpolation='nearest',vmin=0.0,vmax=1, cmap=cmap)
#plt.title(title)
plt.colorbar()

if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=0)
    plt.yticks(tick_marks, target_names)


thresh = cm.max() / 1.5 if normalize else cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    if normalize:
        plt.text(j, i, "{:0.3f}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    else:
        plt.text(j, i, "{:,}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')

plt.savefig('../figures/Fig6_B.svg',format='svg')
plt.show()

In [ ]:
#https://machinelearningmastery.com/precision-recall-and-f-measure-for-imbalanced-classification/

with open('DM_v_MCI_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
    
cm=cm_total
target_names=['DEM','MCI','CN']
title='Confusion matrix'
cmap=None
normalize=True


if cmap is None:
    cmap = plt.get_cmap('Blues')
    newcolors = np.vstack((cmap(np.linspace(0, 0.01, 60)),cmap(np.linspace(0, 1, 128)),
    cmap(np.linspace(0.99, 1, 70))))
    cmap = ListedColormap(newcolors)

if normalize:
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6, 5),dpi=300)
plt.imshow(cm, interpolation='nearest',vmin=0.0,vmax=1, cmap=cmap)
#plt.title(title)
plt.colorbar()

if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=0)
    plt.yticks(tick_marks, target_names)


thresh = cm.max() / 1.5 if normalize else cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    if normalize:
        plt.text(j, i, "{:0.3f}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    else:
        plt.text(j, i, "{:,}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')

plt.savefig('../figures/Fig6_C.svg',format='svg')
plt.show()